In [47]:
import os
import sys
from datetime import datetime
import itertools

from importlib import reload
current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')
os.chdir(current_dir)
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)
import plotting
import pinns
import infinite
reload(plotting)
reload(pinns)
reload(infinite)
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn
import torch.optim as optim
from pinns import  MLP, init_weights, CoefficientNet, pde_loss_inf, observation_loss_u, observation_loss_k, train_dual_network, build_models, set_seed,run_experiment_inf,build_models_KAN
from pinns import build_models
from infinite import analytical_solution_inf, coefficient_inf, source_term_inf, generate_dataset_inf, evaluate_model_inf
torch.set_default_dtype(torch.float32)
from plotting import plot_histories_comparison

set_seed(1)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [48]:
# Define the 9 KAN configurations from the budget-matching table (Depth L, Width N, Grid G=5)
kan_configurations = [
    {"L": 1, "width": 25, "grid_size": 5},
    {"L": 1, "width": 30, "grid_size": 5},
    {"L": 1, "width": 35, "grid_size": 5},
    {"L": 2, "width": 25, "grid_size": 5},
    {"L": 2, "width": 30, "grid_size": 5},
    {"L": 2, "width": 35, "grid_size": 5},
    {"L": 3, "width": 25, "grid_size": 5},
    {"L": 3, "width": 30, "grid_size": 5},
    {"L": 3, "width": 35, "grid_size": 5},
]

# Loop through each table configuration
for cfg in kan_configurations:
    layers = cfg["L"]
    width = cfg["width"]
    grid_sz = cfg["grid_size"]
    
    print(f"\n--- Running KAN Experiment: Layers (L)={layers}, Width (N)={width}, Grid={grid_sz} ---")
    
    try:
        err_u, err_k, compute_time = run_experiment_inf(
            model_type="KAN",
            hidden_layers=layers,      # Swept via table (L = 1, 2, 3)
            hidden_units=width,        # Swept via table (N = 25, 30, 35)
            grid_size=grid_sz,         # Fixed at 5 per table spec
            adam_lr=1e-3,              
            device=device,
            adam_iters=200,            # Adjust as needed
            lbfgs_iters=200,           # Adjust as needed
            results_dir="results_accuracy_efficiency_2",
        )
        print(f"Success! Time: {compute_time:.2f}s | Err U: {err_u:.3e} | Err K: {err_k:.3e}")
        
    except Exception as e:
        print(f"Experiment failed for Layers={layers}, Width={width}, Grid={grid_sz} with error: {e}")


--- Running KAN Experiment: Layers (L)=1, Width (N)=25, Grid=5 ---

[KAN] L=1, N=25 | Params: 1,500 | Mean Err: 1.302e-01 | Saved to 'results_accuracy_efficiency_2/'.
Success! Time: 17.37s | Err U: 1.251e-01 | Err K: 1.353e-01

--- Running KAN Experiment: Layers (L)=1, Width (N)=30, Grid=5 ---

[KAN] L=1, N=30 | Params: 1,800 | Mean Err: 8.505e-02 | Saved to 'results_accuracy_efficiency_2/'.
Success! Time: 16.15s | Err U: 1.432e-01 | Err K: 2.686e-02

--- Running KAN Experiment: Layers (L)=1, Width (N)=35, Grid=5 ---

[KAN] L=1, N=35 | Params: 2,100 | Mean Err: 1.042e-01 | Saved to 'results_accuracy_efficiency_2/'.
Success! Time: 16.04s | Err U: 8.857e-02 | Err K: 1.198e-01

--- Running KAN Experiment: Layers (L)=2, Width (N)=25, Grid=5 ---

[KAN] L=2, N=25 | Params: 14,000 | Mean Err: 6.337e-02 | Saved to 'results_accuracy_efficiency_2/'.
Success! Time: 22.87s | Err U: 9.771e-02 | Err K: 2.904e-02

--- Running KAN Experiment: Layers (L)=2, Width (N)=30, Grid=5 ---

[KAN] L=2, N=30 | 